## Calculating the Units for Multi-family Homes across California

This notebook utilizes parcel data, Zillow data, and building data to calculate missing multi-family unit data. Residential parcels are subset to calculate building volume and create a regression that gives best approximations of multi-family home data. Single family home volume is additionally calculated. This unit data is utilized in calculating hosting capacity to assess where limited distributed energy resources exist across California.



In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

ERROR 1: PROJ: proj_create_from_database: Open of /Users/sarak/.conda/envs/electrigrid-env/share/proj failed


In [ ]:
# os.environ['PROJ_LIB'] = '/opt/anaconda3/share/proj'

In [3]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

### Parcel data

The analysis relies on parcels to calculate which buildings are assigned to which units. Some buildings intersect with more than one parcel which is kept in both calculations would duplicate the volume of thse homes and cause erroneous unit calculations. Hence a unique parcel number is required to ensure that the calculations accurate.

In [ ]:
# # load parcels 
# parcels = gpd.read_parquet("../../../../../../capstone/electrigrid/data/parcels/parcels_final.parquet").to_crs(epsg=4326)

# load from electrigrid complete data
parcels = gpd.read_parquet("electrigrid_complete_data/parcel_data/parcels_final.parquet").to_crs(epsg=4326)

In [5]:
# investigate duplicated parcel numbers
#parcels[parcels['PARNO'].duplicated()]


The parcel number should have been unique to each parcel. As shown above there are 525,130 parcel numbers that are not unique. Instead we'll assume that each row is a different parcel and assign a unique id from the index. 

In [6]:
# make an id column in the parcel dataframe to use as the unique id
parcels['parcel_ID'] = parcels.index

In [110]:
parcels.head()

,parcel_ID,geometry
0,0,"MULTIPOLYGON Z (((-122.12384 37.75571 0.00000,..."
1,1,"MULTIPOLYGON Z (((-122.12504 37.75429 0.00000,..."
2,2,"MULTIPOLYGON Z (((-122.12635 37.75366 0.00000,..."
3,3,"MULTIPOLYGON Z (((-122.12243 37.75165 0.00000,..."
4,4,"MULTIPOLYGON Z (((-122.12829 37.76110 0.00000,..."


### Zillow data

In [ ]:
# load data from electrigrid complete data 
# import Zillow data 
zillow = gpd.read_parquet("electrigrid_complete_data/residential_data/zillow.parquet").to_crs(epsg=4326)

# read in ca boundary to clip zillow
ca_boundary = gpd.read_file("electrigrid_complete_data/ca_state_boundary.geojson").to_crs(epsg=4326)

# clip Zillow to california 
zillow = zillow.clip(ca_boundary)

In [ ]:
# # import Zillow data 
# zillow = gpd.read_parquet("../../../../../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

# # read in ca boundary to clip zillow
# ca_boundary = gpd.read_file("../../../../../../capstone/electrigrid/data/ca_state_boundary.geojson").to_crs(epsg=4326)

# # clip Zillow to california 
# zillow = zillow.clip(ca_boundary)

In [111]:
print(f"Number of Zillow points", zillow.shape[0])

Number of Zillow points 10012568


We expect to get the same number of homes at the end of unit regression calculation.

### Building Footprint data

Access: https://sat-io.earthengine.app/view/gba

In [ ]:
# # import building footprint 
# building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

# import building footprint from electrigrid complete data
building = gpd.read_parquet("electrigrid_complete_data/residential_data/buildings_ca.parquet").to_crs(epsg=4326)

# Unit-regression for multi-family homes and volume calculations for all homes

 Check that subsetting sums up to the total Zillow sum. 


In [13]:
# the analysis only requires the parcel_id and geometry  
parcels = parcels[['parcel_ID', 'geometry']]

## SUBSET ALL THE DATA TYPES
## select only multi-family data from Zillow
zillow_multi_raw = zillow[zillow['type'] == "Multi"]
zillow_multi_raw = zillow_multi_raw[zillow_multi_raw['code'] != "RR106"]

# select the single family home data 
zillow_single_raw = zillow[zillow['type'] == "Single"]

# select the condo data 
zillow_condos_raw = zillow[zillow['code'] == "RR106"]

# check to ensure all of the subsets are accurate, these all add up to the total of the zillow 
assert zillow_multi_raw.shape[0] + zillow_single_raw.shape[0] + zillow_condos_raw.shape[0] == zillow.shape[0]

# set a zillow index column so we can track how much zillow data we lose
zillow_multi_raw['zillow_index'] = zillow_multi_raw.index

In [14]:
print(f"There are ",zillow_multi_raw.shape[0], "multi-family zillow.")

There are  603425 multi-family zillow.


In [15]:
zillow_multi_raw.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,zillow_index
8103273,Multi,NaN,NaN,fossil,central,None,4.0,273409.0,living,2548.0,1428589,06025012001,h438,Others,RI103,POINT (-115.48696 32.66592),8103273
8103323,Multi,NaN,NaN,None,None,I,3.0,230000.0,living,3212.0,1428640,06025012001,h438,Others,RI102,POINT (-115.48582 32.66600),8103323
8103319,Multi,NaN,NaN,elec,room,None,2.0,48239.0,living,1682.0,1428636,06025012001,h438,Others,RI101,POINT (-115.48535 32.66603),8103319
8103315,Multi,NaN,NaN,None,None,None,2.0,48770.0,living,1696.0,1428632,06025012001,h438,Others,RI101,POINT (-115.48459 32.66605),8103315
8103242,Multi,NaN,NaN,None,None,None,3.0,74426.0,living,1732.0,1428557,06025012001,h438,Others,RI101,POINT (-115.48818 32.66606),8103242


#### Valid parcel subset

Subset to parcels with multi-family homes.

In [16]:
## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi_raw, predicate="contains")
## output: each parcel where the a multi_zillow point is within it (one-to-many relationship)

# confirm that joining with Zillow decreases the number of parcels
assert len(valid_parcels) < len(parcels)

# drop units as we will add back summed units per parcel after
valid_parcels = valid_parcels.drop(['unit', 'index_right'], axis = 1)

# sum number of units per parcel
units_sum = valid_parcels.sjoin(zillow_multi_raw, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
valid_parcels = valid_parcels.join(units_sum)
valid_parcels.rename(columns={"unit":"total_unit"}, inplace=True)
## for each parcel we should have the total number of units but this means that every time a parcel appears total units appear 

# drop duplicates based on parcel
valid_parcels = valid_parcels.drop_duplicates(subset = 'parcel_ID', keep = 'first')

In [17]:
valid_parcels.head()

,parcel_ID,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit
92,92,"MULTIPOLYGON Z (((-122.11883 37.75180 0.00000,...",Multi,1970.0,3.0,None,None,O,1447455.0,living,2621.0,135866,06001409900,262,PGE/SCE,RI101,123790,2.0
212,212,"MULTIPOLYGON Z (((-122.17728 37.72609 0.00000,...",Multi,1965.0,6.0,None,None,I,518707.0,living,3038.0,113661,06001409200,568,PGE/SCE,RI103,103006,4.0
231,231,"MULTIPOLYGON Z (((-122.12955 37.75105 0.00000,...",Multi,1955.0,7.0,None,None,O,1335766.0,living,4473.0,135372,06001410000,262,PGE/SCE,RI101,123305,2.0
300,300,"MULTIPOLYGON Z (((-122.25473 37.81431 0.00000,...",Multi,1966.0,36.0,None,None,None,8022880.0,living,24529.0,3702,06001403600,574,PGE/SCE,RI104,2970,26.0
318,318,"MULTIPOLYGON Z (((-122.28456 37.81261 0.00000,...",Multi,1905.0,4.0,None,None,None,270454.0,living,2625.0,162813,06001402400,468,PGE/SCE,RI101,148788,2.0


In [102]:
len(valid_parcels)

621203

In [18]:
print(f"We lose ", len(zillow_multi_raw['zillow_index'].unique()) - len(valid_parcels['zillow_index'].unique()), "homes when assigning to parcels.")

We lose  22481 homes when assigning to parcels.


This could be new developments. The parcel data being from 2014 does not capture all of the current development. These missing homes could be new developments. 
**Assign these homes to their nearest neighbors to calculate their volumes.**

In [19]:
# find the multi-family homes that were not assigned to a parcel
multi_noparcel = zillow_multi_raw[~zillow_multi_raw.zillow_index.isin(valid_parcels['zillow_index'])]

len(multi_noparcel)

22481

As a reminder, `multi_noparcel` contains multi-family Zillow observations that are not within any of the parcels in our data.

In [20]:
# adjust to projected CRS for nearest join 
multi_noparcel = multi_noparcel.to_crs("EPSG:6933")
building = building.to_crs("EPSG:6933")

In [21]:
# rename building index
multi_noparcel = multi_noparcel.rename(columns = {'index__right':'building_index'})

In [22]:
multi_noparcel.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,zillow_index
8102815,Multi,NaN,NaN,fossil,central,None,2.0,79109.0,living,1275.0,1428036,06025012101,h438,Others,RI101,POINT (-11143921.954 3950680.272),8102815
8103109,Multi,NaN,NaN,fossil,central,O,2.0,151284.0,living,2220.0,1428391,06025012001,h438,Others,RI101,POINT (-11143003.210 3950692.232),8103109
8114429,Multi,NaN,NaN,fossil,central,None,NaN,323000.0,living,2976.0,1440901,06025012001,h438,Others,RI102,POINT (-11143076.926 3950814.706),8114429
8103864,Multi,NaN,NaN,fossil,central,O,1.0,248335.0,living,2350.0,1429183,06025012202,h462,Others,RI101,POINT (-11145813.187 3951333.165),8103864
8102127,Multi,NaN,NaN,fossil,central,None,2.0,275400.0,living,1323.0,1427244,06025012102,h438,Others,RI101,POINT (-11143936.715 3951602.990),8102127


In [23]:
# Join Zillow points to the nearest building geometry
multi_noparcel = gpd.sjoin_nearest(multi_noparcel,
                                        building,
                                        how="left", 
                                        lsuffix='_left', 
                                        rsuffix='_right',
                                        distance_col='dist_to_building')

In [24]:
# check length after join to buildings
len(multi_noparcel)

24251

In [25]:
# drop bbox column -- the duplicated function doesn't like dictionaries
multi_noparcel = multi_noparcel.drop('bbox', axis =1)

multi_noparcel.duplicated().sum()

1770

Duplicates created if buildings are equidistant from zillow points.

In [26]:
# drop duplicates
multi_noparcel = multi_noparcel.drop_duplicates()

#### Buildings within multi-family parcels

Assign buildings to parcels which already includes Zillow data. 

In [27]:
# revert building CRS
building = building.to_crs('EPSG:4326')
multi_noparcel = multi_noparcel.to_crs('EPSG:4326')

In [28]:
# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(valid_parcels, predicate="intersects")

# drop the unnecessary column 
valid_buildings = valid_buildings.drop(columns = 'index_right', axis = 1)

# confirm that joining with Zillow decreased the number of buildings
assert len(valid_buildings) < len(building)

In [29]:
valid_buildings.head()

,source,id,height,var,region,bbox,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"{'xmax': -121.54639413033861, 'xmin': -121.546...","POLYGON ((-121.54641 40.08095, -121.54639 40.0...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0
18575,ms,UnitedStates_023010013_3773,3.229227,0.503842,USA,"{'xmax': -121.54630000377989, 'xmin': -121.546...","POLYGON ((-121.54630 40.08066, -121.54630 40.0...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0
19630,ms,UnitedStates_023010012_33729,4.058523,0.654597,USA,"{'xmax': -121.87064520706676, 'xmin': -121.870...","POLYGON ((-121.87074 40.43491, -121.87065 40.4...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0
19631,ms,UnitedStates_023010012_16394,6.213455,0.869858,USA,"{'xmax': -121.8706356078727, 'xmin': -121.8707...","POLYGON ((-121.87072 40.43514, -121.87066 40.4...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0
19632,ms,UnitedStates_023010012_32724,2.698684,0.152872,USA,"{'xmax': -121.8708256701087, 'xmin': -121.8709...","POLYGON ((-121.87083 40.43542, -121.87090 40.4...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0


In [97]:
len(valid_buildings)

6913712

In [30]:
print(f"We lose ", len(valid_parcels['parcel_ID'].unique()) - len(valid_buildings['parcel_ID'].unique()), "parcels when assigning buildings")

We lose  9677 parcels when assigning buildings


**Could there potentially be some parcels with no buildings? Or have missing building data?"**

In [31]:
print(f"From raw zillow to buidlings we lose", len(zillow_multi_raw['zillow_index'].unique()) - len(valid_buildings['zillow_index'].unique()) ,"zillow points")
print(f"From parcels to buildings we lose", len(valid_parcels['zillow_index'].unique()) - len(valid_buildings['zillow_index'].unique()), "zillow points")

From raw zillow to buidlings we lose 31833 zillow points
From parcels to buildings we lose 9352 zillow points


### Remove parcel duplicates

Buildings can straddle more than one parcel. This causes the volume to be utilized in the regression more than once. To avoid this we assign the building to the parcel where more of the building area is.

In [32]:
# change the crs to a projected crs
building_zillow_parcels = valid_buildings.to_crs("EPSG:6933")
valid_parcels_proj = valid_parcels.to_crs("EPSG:6933").set_index('parcel_ID')

In [33]:
len(building_zillow_parcels)

6913712

In [34]:
len(valid_parcels_proj)

621203

In [35]:
# to fix geometry problems from the crs change
building_zillow_parcels.geometry = building_zillow_parcels.geometry.buffer(0)
valid_parcels_proj.geometry = valid_parcels_proj.geometry.buffer(0)

In [36]:
# calculate intersection area for each building-parcel pair
building_zillow_parcels["intersection_area"] = (
    building_zillow_parcels.geometry.values
    .intersection(valid_parcels_proj.loc[building_zillow_parcels['parcel_ID']].geometry.values).area)

In [37]:
len(building_zillow_parcels)

6913712

In [38]:
# keep only the parcel with the largest overlap per building
building_zillow_parcels_merge = (
    building_zillow_parcels
    .sort_values("intersection_area", ascending=False)
    .drop_duplicates(subset=["id", "parcel_ID"], keep="first") # fix -- keep per building, parcel pair
    .drop(columns="intersection_area"))

In [39]:
len(building_zillow_parcels_merge)

6889932

In theory, this value should be greater than the number of multi-family homes in the zillow data, and smaller than the number of `valid_buildings`.

In [40]:
len(building_zillow_parcels_merge) > len(zillow_multi_raw)

True

In [41]:
len(building_zillow_parcels_merge) < len(valid_buildings)

True

In [42]:
building_zillow_parcels = building_zillow_parcels.drop('bbox', axis =1)

building_zillow_parcels.duplicated().sum()

23775

In [43]:
building_zillow_parcels.head()

,source,id,height,var,region,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,intersection_area
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"POLYGON ((-11727561.053 4715034.549, -11727570...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,62.973518
18575,ms,UnitedStates_023010013_3773,3.229227,0.503842,USA,"POLYGON ((-11727550.366 4715005.976, -11727556...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,44.274674
19630,ms,UnitedStates_023010012_33729,4.058523,0.654597,USA,"POLYGON ((-11758854.286 4749688.155, -11758858...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,9.786246
19631,ms,UnitedStates_023010012_16394,6.213455,0.869858,USA,"POLYGON ((-11758852.127 4749710.399, -11758849...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,59.785924
19632,ms,UnitedStates_023010012_32724,2.698684,0.152872,USA,"POLYGON ((-11758862.640 4749738.070, -11758867...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,74.050334


In [44]:
len(zillow_multi_raw)

603425

In [45]:
print(f"{len(building_zillow_parcels)} vs {len(building_zillow_parcels_merge)}")

6913712 vs 6889932


In [46]:
print(f"Difference that area method makes: {len(building_zillow_parcels) - len(building_zillow_parcels_merge)}")
print(f"Duplicates in original data: {building_zillow_parcels.duplicated().sum()}")

Difference that area method makes: 23780
Duplicates in original data: 23775


So our merge removes 5 buildings that were originally not duplicated...idk how this can be, but better than before!

In [47]:
print(f"We lose",len(valid_buildings['zillow_index'].unique()) - len(building_zillow_parcels['zillow_index'].unique()), "zillow points when trying to remove duplicates." )

We lose 0 zillow points when trying to remove duplicates.


### Calculate volume

**Using valid buildings since we lose so much data when trying to account for the duplicates**

The regression runs on the assumption that volume is linearly correlated with the number of units. First we calculate volume using the area of buildings. Then we run a regression to calculate the number of units where missing in the data.

We currently have three different data subsets:
- multi-family homes (assigned to building through parcels or nearest join)
- single-family homes
- condos

We will calculate volume for each.

#### Multi-family homes

We will concat the `multi_noparcel` and ` valid_buildings` data frames -- `multi_noparcel` will contains NA values in the parcel column.

In [48]:
# have to rename unit column to match "total_unit" in valid_buildings
multi_noparcel = multi_noparcel.rename(columns = {"unit":"total_unit"})

In [49]:
# and reproject data frame to crs with meters as units
building_m = pd.concat([valid_buildings, multi_noparcel]).to_crs("EPSG:6933") 

assert len(valid_buildings) + len(multi_noparcel) == len(building_m)

In [50]:
building_m.head()

,source,id,height,var,region,bbox,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,index__right,dist_to_building
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"{'xmax': -121.54639413033861, 'xmin': -121.546...","POLYGON ((-11727561.053 4715034.549, -11727559...",480115.0,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,NaN,NaN
18575,ms,UnitedStates_023010013_3773,3.229227,0.503842,USA,"{'xmax': -121.54630000377989, 'xmin': -121.546...","POLYGON ((-11727550.366 4715005.976, -11727550...",480115.0,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,NaN,NaN
19630,ms,UnitedStates_023010012_33729,4.058523,0.654597,USA,"{'xmax': -121.87064520706676, 'xmin': -121.870...","POLYGON ((-11758854.286 4749688.155, -11758845...",10631193.0,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,NaN,NaN
19631,ms,UnitedStates_023010012_16394,6.213455,0.869858,USA,"{'xmax': -121.8706356078727, 'xmin': -121.8707...","POLYGON ((-11758852.127 4749710.399, -11758847...",10631193.0,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,NaN,NaN
19632,ms,UnitedStates_023010012_32724,2.698684,0.152872,USA,"{'xmax': -121.8708256701087, 'xmin': -121.8709...","POLYGON ((-11758862.640 4749738.070, -11758869...",10631193.0,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,NaN,NaN


In [51]:
## CALCULATE VOLUME!

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

### Single-family homes

Join Zillow points to buildings for volume calculation.

In [52]:
# switch to projected CRS (for calculating area)
zillow_single = zillow_single_raw.to_crs("EPSG:6933")
building = building.to_crs("EPSG:6933")

# Join single family homes to nearest building (for area and height data)
zillow_single = zillow_single.sjoin(building, how = "left", predicate="within")

# Drop bbox column (creates error) and duplicates 
zillow_single = zillow_single.drop('bbox', axis =1 )
zillow_single = zillow_single.drop_duplicates()

# Save those unmatched to geometries
unmatched_single = zillow_single[zillow_single['index_right'].isna()]

# Remove those unmatched from data
zillow_single = zillow_single[~zillow_single['index_right'].isna()]

# rejoin building geometry
zillow_single['building_geometry'] = zillow_single['index_right'].map(building['geometry'][~building.index.duplicated()])

# set zillow geometry to building geo
zillow_single = zillow_single.set_geometry('building_geometry')

In [101]:
len(unmatched_single) / len(zillow_single_raw) * 100

28.806122971319514

In [107]:
len(unmatched_single)

2369123

In [108]:
len(zillow_single)

8224422

In [53]:
print(f"{len(zillow_single_raw)} vs {len(zillow_single) + len(unmatched_single)}")

8224373 vs 8224422


This join introduces 49 new observations. The world will never know why.

#### Calculate volume!

In [54]:
# create column from polygon area
zillow_single['area_m2'] = zillow_single.geometry.area

# rename height column to be clear about units
zillow_single.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
zillow_single['volume_m3'] = zillow_single['area_m2'] * zillow_single['height_m']

In [55]:
len(zillow_single)

5855299

Concat observations with unmatched geometries (and therefore no volume), back to the data frame.

In [56]:
len(unmatched_single) + len(zillow_single)

8224422

In [57]:
unmatched_single = unmatched_single.rename(columns={"height":"height_m"})

In [58]:
zillow_single = pd.concat([zillow_single, unmatched_single])

In [59]:
len(zillow_single)

8224422

### Condos (same workflow as single family homes)

In [60]:
# switch to projected CRS (for calculating area)
zillow_condos = zillow_condos_raw.to_crs("EPSG:6933")

# Join condos to intersecting buildings (for area and height data)
zillow_condos = zillow_condos.sjoin(building, how = "left", predicate="within")

# Drop bbox column (creates error) and duplicates 
zillow_condos = zillow_condos.drop('bbox', axis =1 )
zillow_condos = zillow_condos.drop_duplicates()

# Save those unmatched to geometries
unmatched_condos = zillow_condos[zillow_condos['index_right'].isna()]

# Remove those unmatched from data
zillow_condos = zillow_condos[~zillow_condos['index_right'].isna()]

zillow_condos['building_geometry'] = zillow_condos['index_right'].map(building['geometry'][~building.index.duplicated()])

# set zillow geometry to building geo
zillow_condos = zillow_condos.set_geometry('building_geometry')

In [61]:
print(f"{len(zillow_condos_raw)} vs {len(zillow_condos) + len(unmatched_condos)}")

1184770 vs 1184771


And this join introduces 1 new condo observation! Again, we will never know why.

#### Calculate volume! For those with matched geometries

In [62]:
# create column from polygon area
zillow_condos['area_m2'] = zillow_condos.geometry.area

# rename height column to be clear about units
zillow_condos.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
zillow_condos['volume_m3'] = zillow_condos['area_m2'] * zillow_condos['height_m']

We will concat the data with unmatched geomtries (so no volume data) back to dataframe.

In [63]:
len(unmatched_condos) + len(zillow_condos)

1184771

In [109]:
len(unmatched_condos)

451021

In [64]:
unmatched_condos = unmatched_condos.rename(columns={"height":"height_m"})

In [65]:
zillow_condos = pd.concat([zillow_condos, unmatched_condos])

In [66]:
len(zillow_condos)

1184771

#### Units Assumption

We are making the assumption that the tax assessor data was correct in its identification of single-family homes and condos, we will assign the `unit` columns in both datasets to be exactly 1 for each observation.

In [67]:
zillow_single['total_unit'] = 1
zillow_condos['total_unit'] = 1

In [112]:
building_w_units = building_m[~building_m['total_unit'].isna()]

In [115]:
len(building_w_units[building_w_units['total_unit'] == 0])

1556882

## The rest of the regression

This part of the workflow only works with the multi-family homes -- the ones that *have* unit data are being used to build the regression, and those with *missing* unit data use that regression to predict the number of units.

In [68]:
# keep only observations with unit data
building_w_units = building_m[~building_m['total_unit'].isna()]

assert building_w_units['total_unit'].isna().sum() == 0

# drop multi-family homes where the total_unit is equal to zero
building_w_units = building_w_units.drop(building_w_units[building_w_units['total_unit'] == 0].index)

# aggregate the volumes by parcel
total_volume_m3 = building_w_units.groupby('parcel_ID')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'parcel_ID')

# some of the homes don't have a parcel number (so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

# remove duplicates from the parcel number so that it doesn't skew the linear regression
building_w_units = building_w_units.drop_duplicates(subset = 'parcel_ID', keep = 'first')

# run model
results = smf.ols('total_unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('total_unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params.iloc[0]
slope = results_clean.params.iloc[1]

In [69]:
results_clean.params.iloc[1]

0.0013897028142429333

In [92]:
len(building_outliers)

94

In [94]:
len(building_w_units)

508572

In [95]:
len(building_m)

6936193

In [99]:
len(building_w_units) / len(building_m)

0.0733214891800156

In [70]:
slope

0.0013897028142429333

In [71]:
intercept

7.445143530374704

#### Utilize the regression to estimate units where they're missing  

In [72]:
# extract just the multi-family homes where unit info is missing
missing_units = building_m[(building_m['total_unit'].isna()) | (building_m['total_unit'] == 0)]

# aggregate the volumes for the unit regression by parcel
missing_total_volume_m3 = missing_units.groupby('ID')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
missing_total_volume_m3 = pd.DataFrame(missing_total_volume_m3)

# rename the column to missing_total_volume_m3
missing_total_volume_m3 = missing_total_volume_m3.rename(columns = {'volume_m3' : 'missing_total_volume_m3'})

# add the missing total volume to the buildings dataframe
missing_units = missing_units.join(missing_total_volume_m3, on = 'parcel_ID')

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace anywhere the missing_total_volume is missing (fill in for the outliers since total_volume was computed before)
missing_outlier_units['missing_total_volume_m3'] = missing_outlier_units['missing_total_volume_m3'].fillna(missing_outlier_units['total_volume_m3'])

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('total_unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

# utilize the values from the regression to calculate units
missing_outlier_units_pred['total_unit'] = round(intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope)

# calculate each building's share of parcel volume
missing_outlier_units_pred['volume_share'] = (
    missing_outlier_units_pred['volume_m3'] / 
    missing_outlier_units_pred['missing_total_volume_m3'].replace(0, np.nan))

# distribute units proportionally, rounding to nearest int
missing_outlier_units_pred['total_unit'] = round(
    (intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope) 
    * missing_outlier_units_pred['volume_share'])

# combine multi-family homes data frames
multi_complete = pd.concat([building_units_clean, missing_outlier_units_pred]).to_crs(zillow.crs)

# fill the total_volume for those with the missing_total_volume_m3 and rename to total_volume_m3
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# fill the total_volume for those with the missing_total_volume_m3 
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# drop unnecessary columns 
multi_complete = multi_complete.drop(['residual','total_volume_m3'], axis = 1)

# rename to total_volume_m3
multi_complete = multi_complete.rename(columns = {'missing_total_volume_m3' : 'total_volume_m3'})

### Cap `total_unit` Value

For some outliers with high building volumes, our regression predicts an unreasonable number of units. We will cap our predictions at the 99.9th percentile of the non-predicted units. 

In [73]:
value = np.percentile(building_w_units['total_unit'], 99.9)

In [74]:
value

1788.0

In [75]:
print(f"This would replace predicted unit values for {len(multi_complete[multi_complete['total_unit'] > value])} homes.")

This would replace predicted unit values for 296 homes.


In [76]:
# Set up if else statement
multi_complete['total_unit'] = np.where(multi_complete['total_unit'] > value, value, multi_complete['total_unit'])

### Create points from the building polygons

Single family homes and condos are points not polygons. For uniformity and hosting capacity calculation set the points as centroids to the parcels.

#### All multi-family homes

In [77]:
# projected CRS
multi_complete = multi_complete.to_crs("EPSG:6933")

# find centroid
multi_complete['centroid'] =  multi_complete.geometry.centroid

In [78]:
#### Single-family homes and condos

zillow_single['centroid'] = zillow_single.geometry.centroid

zillow_condos['centroid'] = zillow_condos.geometry.centroid

In [79]:
# If geometry for centroid is missing, replace with Zillow point
zillow_single['centroid'] = np.where(zillow_single['building_geometry'].isna(), zillow_single['geometry'], zillow_single['centroid'])
zillow_condos['centroid'] = np.where(zillow_condos['building_geometry'].isna(), zillow_condos['geometry'], zillow_condos['centroid'])

In [80]:
multi_complete.head()

,source,id,height_m,var,region,bbox,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,index__right,dist_to_building,area_m2,volume_m3,total_volume_m3,volume_share,centroid
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"{'xmax': -121.54639413033861, 'xmin': -121.546...","POLYGON ((-11727561.053 4715034.549, -11727559...",480115.0,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,NaN,NaN,62.973518,283.905634,853.757240,NaN,POINT (-11727564.987 4715038.471)
26238,osm,759299775,1.686291,0.644541,USA,"{'xmax': -121.43016199999998, 'xmin': -121.430...","POLYGON ((-11716354.676 4773227.596, -11716348...",10084995.0,Multi,NaN,1.0,None,None,None,448916.0,living,1064.0,9911222,06089012701,302,PGE/SCE,RI101,6245297,2.0,NaN,NaN,38.333495,64.641441,12930.381426,NaN,POINT (-11716349.663 4773227.774)
30005,ms,UnitedStates_023010010_16,3.319534,0.280181,USA,"{'xmax': -121.69706788384876, 'xmin': -121.697...","POLYGON ((-11742113.188 4792054.055, -11742112...",10001573.0,Multi,NaN,3.0,None,None,I,96001.0,living,1344.0,9910184,06089012701,357,PGE/SCE,RI101,6244425,1.0,NaN,NaN,283.595485,941.404872,7037.163187,NaN,POINT (-11742109.108 4792044.168)
31048,ms,UnitedStates_023010010_1720,7.291415,0.815212,USA,"{'xmax': -121.91076675749248, 'xmin': -121.910...","POLYGON ((-11762731.093 4796438.800, -11762721...",10086613.0,Multi,NaN,3.0,None,None,None,176731.0,living,1476.0,9908514,06089012606,411,PGE/SCE,RI101,6242951,3.0,NaN,NaN,91.268270,665.474852,7736.379409,NaN,POINT (-11762723.750 4796440.688)
31157,ms,UnitedStates_021232233_5134,6.432827,2.174101,USA,"{'xmax': -121.63049440002435, 'xmin': -121.630...","POLYGON ((-11735680.314 4804466.517, -11735681...",10000041.0,Multi,NaN,2.0,None,None,None,357000.0,living,3456.0,9908031,06089012701,623,PGE/SCE,RI101,6242607,4.0,NaN,NaN,316.484363,2035.889151,7581.420633,NaN,POINT (-11735686.517 4804476.272)


In [81]:
zillow_single.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,index_right,source,id,height_m,var,region,building_geometry,area_m2,volume_m3,total_unit,centroid
8103247,Single,NaN,NaN,None,None,O,NaN,44222.0,None,NaN,1428562,06025012001,h438,Others,RR101,POINT (-11143051.551 3950296.232),126632.0,ms,UnitedStates_023013231_19488,3.882336,0.875427,USA,"POLYGON ((-11791660.955 4761459.170, -11791659...",441.261902,1713.126821,1,POINT (-11791650.196110915 4761448.595561568)
8103246,Single,NaN,NaN,None,None,O,NaN,114544.0,None,NaN,1428561,06025012001,h438,Others,RR101,POINT (-11143029.166 3950297.095),126633.0,ms,UnitedStates_023013231_18983,3.750597,1.485676,USA,"POLYGON ((-11791653.930 4761476.792, -11791671...",188.696046,707.722870,1,POINT (-11791662.867700228 4761471.5684387665)
8103224,Single,NaN,NaN,elec,room,O,1.0,62187.0,living,1294.0,1428535,06025012101,h438,Others,RR101,POINT (-11143310.810 3950308.418),126643.0,ms,UnitedStates_023013231_28389,3.657947,1.622700,USA,"POLYGON ((-11791258.951 4761577.589, -11791259...",61.172565,223.765977,1,POINT (-11791263.496017637 4761580.995394463)
8103268,Single,NaN,NaN,fossil,room,None,NaN,182272.0,living,1502.0,1428584,06025012001,h438,Others,RR101,POINT (-11142828.957 3950312.187),126593.0,ms,UnitedStates_023013231_32434,4.868192,0.718464,USA,"POLYGON ((-11790415.768 4761978.968, -11790388...",283.843284,1381.803523,1,POINT (-11790402.77557273 4761985.25602834)
8103274,Single,NaN,NaN,elec,room,None,1.0,193520.0,living,884.0,1428590,06025012001,h438,Others,RR101,POINT (-11142923.320 3950317.148),126582.0,ms,UnitedStates_023013231_42636,3.861226,1.579459,USA,"POLYGON ((-11790304.149 4761976.186, -11790304...",284.937764,1100.208989,1,POINT (-11790315.619758112 4761982.350162581)


In [82]:
zillow_condos.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,index_right,source,id,height_m,var,region,building_geometry,area_m2,volume_m3,total_unit,centroid
8104600,Multi,NaN,NaN,fossil,central,None,NaN,159515.0,living,1452.0,1429921,06025012003,h279,Others,RR106,POINT (-11142279.850 3951812.433),124103.0,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"POLYGON ((-11794456.181 4765154.432, -11794449...",39.415792,113.70351,1,POINT (-11794453.024076462 4765157.576883913)
8104552,Multi,NaN,NaN,fossil,central,O,NaN,229500.0,living,1452.0,1429873,06025012003,h279,Others,RR106,POINT (-11142279.850 3951812.433),124103.0,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"POLYGON ((-11794456.181 4765154.432, -11794449...",39.415792,113.70351,1,POINT (-11794453.024076462 4765157.576883913)
8104601,Multi,NaN,NaN,fossil,central,None,NaN,159291.0,living,1276.0,1429922,06025012003,h279,Others,RR106,POINT (-11142279.850 3951812.433),124103.0,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"POLYGON ((-11794456.181 4765154.432, -11794449...",39.415792,113.70351,1,POINT (-11794453.024076462 4765157.576883913)
8104575,Multi,NaN,NaN,fossil,central,I,NaN,204978.0,living,1452.0,1429896,06025012003,h279,Others,RR106,POINT (-11142279.850 3951812.433),124103.0,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"POLYGON ((-11794456.181 4765154.432, -11794449...",39.415792,113.70351,1,POINT (-11794453.024076462 4765157.576883913)
8104622,Multi,NaN,NaN,fossil,central,None,NaN,129170.0,living,1452.0,1429943,06025012003,h279,Others,RR106,POINT (-11142279.850 3951812.433),124103.0,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"POLYGON ((-11794456.181 4765154.432, -11794449...",39.415792,113.70351,1,POINT (-11794453.024076462 4765157.576883913)


In [83]:
print(f"Our original zillow data had {len(zillow)}. We have a total of {len(multi_complete) + len(zillow_single) + len(zillow_condos)} observations in our current data. This is a difference of {(len(zillow) - (len(multi_complete) + len(zillow_single) + len(zillow_condos)))} observations.")

Our original zillow data had 10012568. We have a total of 11489502 observations in our current data. This is a difference of -1476934 observations.


In [84]:
print(f"We lost {len(zillow_single_raw) - len(zillow_single)} observations in our single-family homes. This is most likely due to a lack of building footprint data.")

We lost -49 observations in our single-family homes. This is most likely due to a lack of building footprint data.


In [85]:
print(f"We lost {len(zillow_condos_raw) - len(zillow_condos)} observations in our condo data, most likely for similar reasons.")

We lost -1 observations in our condo data, most likely for similar reasons.


In [86]:
print(f"Our multi-family observations increased by {len(multi_complete) - len(zillow_multi_raw)}. This is due to the assignment of individual Zillow points to multiple building footprints.")

Our multi-family observations increased by 1476884. This is due to the assignment of individual Zillow points to multiple building footprints.


## Final Step: Concatenate our three building types into one dataframe

In [87]:
final = pd.concat([multi_complete, zillow_single, zillow_condos])

In [88]:
# Some cleaning up of columns
final = final.drop(['bbox', 'index__right', 'dist_to_building', 'index_right'], axis = 1)

In [89]:
final.head()

,source,id,height_m,var,region,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,area_m2,volume_m3,total_volume_m3,volume_share,centroid,unit,building_geometry
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"POLYGON ((-11727561.053 4715034.549, -11727559...",480115.0,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152.0,1.0,62.973518,283.905634,853.757240,NaN,POINT (-11727564.987130238 4715038.470829107),NaN,None
26238,osm,759299775,1.686291,0.644541,USA,"POLYGON ((-11716354.676 4773227.596, -11716348...",10084995.0,Multi,NaN,1.0,None,None,None,448916.0,living,1064.0,9911222,06089012701,302,PGE/SCE,RI101,6245297.0,2.0,38.333495,64.641441,12930.381426,NaN,POINT (-11716349.662616724 4773227.774179936),NaN,None
30005,ms,UnitedStates_023010010_16,3.319534,0.280181,USA,"POLYGON ((-11742113.188 4792054.055, -11742112...",10001573.0,Multi,NaN,3.0,None,None,I,96001.0,living,1344.0,9910184,06089012701,357,PGE/SCE,RI101,6244425.0,1.0,283.595485,941.404872,7037.163187,NaN,POINT (-11742109.107743217 4792044.167612847),NaN,None
31048,ms,UnitedStates_023010010_1720,7.291415,0.815212,USA,"POLYGON ((-11762731.093 4796438.800, -11762721...",10086613.0,Multi,NaN,3.0,None,None,None,176731.0,living,1476.0,9908514,06089012606,411,PGE/SCE,RI101,6242951.0,3.0,91.268270,665.474852,7736.379409,NaN,POINT (-11762723.750017494 4796440.688433249),NaN,None
31157,ms,UnitedStates_021232233_5134,6.432827,2.174101,USA,"POLYGON ((-11735680.314 4804466.517, -11735681...",10000041.0,Multi,NaN,2.0,None,None,None,357000.0,living,3456.0,9908031,06089012701,623,PGE/SCE,RI101,6242607.0,4.0,316.484363,2035.889151,7581.420633,NaN,POINT (-11735686.516882697 4804476.272443244),NaN,None


In [90]:
# Convert centroid column to parquet-legible geometry
final['centroid'] = gpd.GeoSeries(final['centroid']).to_wkt()

In [103]:
len(final)

11489502

In [104]:
len(final[final['type'] == "Single"])

8224422

In [105]:
len(final[final['code'] == "RR106"])

1184771

In [106]:
len(multi_complete)

2080309

In [91]:
# SAVE
# final.to_parquet("final_building.parquet")